# V2.5.1 — Risk Feature Test: does adding high_volatility_prob make the price model more accurate?

**Controlled experiment**:
- Data: the same risk-enhanced dataset (`V2.5.1_15min_Risk_Enhanced_Dataset.csv`)
- Split: the same chronological 80/20 split
- Hyperparameters: the same set (the Optuna best parameters found in V2.5.2)
- **Only difference**: whether to include the `high_volatility_prob` feature

We compare "with / without the risk feature" for both XGBoost and LightGBM, and check whether MAE / RMSE / R² improve.

In [9]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# Load the "risk-enhanced" dataset (already contains the high_volatility_prob column)
df = pd.read_csv('../data/convertData/V2.5.1_15min_Risk_Enhanced_Dataset.csv')
print('Raw shape:', df.shape)

# Timezone handling: mixed timezones (winter +02:00 / summer +03:00) -> convert to UTC first, then to Helsinki
df['datetime'] = pd.to_datetime(df['datetime'], utc=True).dt.tz_convert('Europe/Helsinki')
df = df.sort_values('datetime').reset_index(drop=True)
print('Number of columns:', len(df.columns))
print('Contains new feature high_volatility_prob:', 'high_volatility_prob' in df.columns)

Raw shape: (105193, 54)
Number of columns: 54
Contains new feature high_volatility_prob: True


In [10]:
# ═══════════════════════════════════════════════════════════════════════
# Build two feature matrices (the key to a controlled experiment):
# - baseline: the 49 original features, WITHOUT any risk columns
# - enhanced: the 49 original features + high_volatility_prob (50 features)
# NOTE: price_roll_std_6h and is_high_volatility are "answers derived from price",
#       so including them as features would leak information; both versions drop them.
# ═══════════════════════════════════════════════════════════════════════
risk_cols = ['price_roll_std_6h', 'is_high_volatility', 'high_volatility_prob']

baseline_cols = [c for c in df.columns if c not in ['price', 'datetime'] + risk_cols]
enhanced_cols = baseline_cols + ['high_volatility_prob']

X_base = df[baseline_cols]          # 49 features
X_enh  = df[enhanced_cols]          # 50 features (one extra: high_volatility_prob)
y = df['price']

print(f'baseline: {len(baseline_cols)} features')
print(f'enhanced: {len(enhanced_cols)} features')

# The same chronological 80/20 split (both feature sets use the exact same rows)
n = len(df)
test_size = int(n * 0.20)
train_end = n - test_size

X_base_train, X_base_test = X_base.iloc[:train_end], X_base.iloc[train_end:]
X_enh_train,  X_enh_test  = X_enh.iloc[:train_end],  X_enh.iloc[train_end:]
y_train, y_test = y.iloc[:train_end], y.iloc[train_end:]
print(f'Train: {X_base_train.shape[0]}  Test: {X_base_test.shape[0]}')

baseline: 49 features
enhanced: 50 features
Train: 84155  Test: 21038


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Use the Optuna best parameters from V2.5.2 (fixed, no new search)
# 2 models x 2 feature sets -> train 4 models in total
# ═══════════════════════════════════════════════════════════════════════
xgb_params = dict(
    objective='reg:absoluteerror', n_estimators=2000,
    learning_rate=0.0590, max_depth=7, min_child_weight=21,
    subsample=0.7268, colsample_bytree=0.9753,
    reg_lambda=0.3616, reg_alpha=0.6782, random_state=42,
)

lgb_params = dict(
    objective='regression_l1', n_estimators=2000,
    learning_rate=0.0168, num_leaves=313, min_child_samples=36,
    subsample=0.9915, colsample_bytree=0.9784,
    reg_lambda=0.2826, reg_alpha=0.2350, random_state=42,
)

def make_xgb():
    return XGBRegressor(**xgb_params, verbosity=0)

def make_lgb():
    return lgb.LGBMRegressor(**lgb_params, verbose=-1)

def train_eval(make_model, Xtr, ytr, Xte, yte):
    """Train one model and evaluate it on the test set, returning (MAE, RMSE, R2)."""
    m = make_model()
    m.fit(Xtr, ytr)
    p = m.predict(Xte)
    return (mean_absolute_error(yte, p),
            np.sqrt(mean_squared_error(yte, p)),
            r2_score(yte, p))

results = {
    'XGBoost baseline':  train_eval(make_xgb, X_base_train, y_train, X_base_test, y_test),
    'XGBoost +risk':     train_eval(make_xgb, X_enh_train,  y_train, X_enh_test,  y_test),
    'LightGBM baseline': train_eval(make_lgb, X_base_train, y_train, X_base_test, y_test),
    'LightGBM +risk':    train_eval(make_lgb, X_enh_train,  y_train, X_enh_test,  y_test),
}

print('All 4 models trained.')

All 4 models trained.


In [12]:
# ═══════════════════════════════════════════════════════════════════════
# Comparison table + verdict
# ═══════════════════════════════════════════════════════════════════════
comp = pd.DataFrame(results, index=['MAE', 'RMSE', 'R2']).T.round(4)
print(comp)

print('\nVerdict (look at MAE; lower is better):')
for name in ['XGBoost', 'LightGBM']:
    base_mae = comp.loc[f'{name} baseline', 'MAE']
    enh_mae  = comp.loc[f'{name} +risk', 'MAE']
    delta = enh_mae - base_mae
    if delta < 0:
        verdict = 'the risk feature HELPS (MAE decreased)'
    elif delta > 0:
        verdict = 'the risk feature HURTS (MAE increased)'
    else:
        verdict = 'the risk feature makes no difference'
    print(f'{name}: MAE {base_mae:.4f} -> {enh_mae:.4f}  delta {delta:+.4f}  {verdict}')

                      MAE    RMSE      R2
XGBoost baseline   2.7555  8.0842  0.9727
XGBoost +risk      2.7957  8.2468  0.9716
LightGBM baseline  2.7165  8.1066  0.9726
LightGBM +risk     2.7426  8.3181  0.9711

Verdict (look at MAE; lower is better):
XGBoost: MAE 2.7555 -> 2.7957  delta +0.0402  the risk feature HURTS (MAE increased)
LightGBM: MAE 2.7165 -> 2.7426  delta +0.0261  the risk feature HURTS (MAE increased)
